# Data Quality Completeness Check Demo

This notebook demonstrates the data quality completeness checking framework.

## Purpose
Runs completeness checks on all columns of the employees dataset to identify null values and calculate null percentages.

## What it does
* Loads configuration from YAML file containing DQ thresholds
* Reads employee data from CSV
* Executes completeness checks across all columns using the DQ framework
* Compares null percentages against configured warning thresholds
* Saves results to a Delta table for tracking and reporting

## Output
* Completeness report showing null counts and percentages per column
* Results saved to `workspace.default.completeness_report` table

In [0]:
"""Data Quality Completeness Check Execution

This script runs completeness validation checks on the employees dataset.
It uses the DQ framework to identify null values and compare against thresholds.
"""

# Import required libraries
import sys
import logging
import yaml

# Add DQ checks module to path and import check functions
sys.path.append('/Workspace/Repos/maha.b.lakshmi@gmail.com/data-quality-testing/src/checks')
from dq_checks import check_completeness, check_completeness_all


config_path = "/Workspace/Repos/maha.b.lakshmi@gmail.com/data-quality-testing/src/config/config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Load employee test data from CSV
employees = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Workspace/Repos/maha.b.lakshmi@gmail.com/data-quality-testing/tests/test_data/employees.csv")

# Run completeness checks on all columns and compare against threshold
results = check_completeness_all(employees, max_null_pct=config["thresholds"]["null_warning_pct"])

# Convert results to DataFrame and display
results_df = spark.createDataFrame(results)
results_df.display()

# Save completeness report to Delta table for tracking and analysis
results_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.completeness_report")

In [0]:
# Option 1: Read using Spark SQL
df = spark.table("workspace.default.completeness_report")
df.display()
